# DSA with CODI's training recipe

Trains one DSA arm with the parts of CODI's GPT-2 recipe that DSA shares, then evaluates it
(`docs/CODI_RECIPE.md`):
- LoRA rank 128 at learning rate 3e-3, with warmup and a cosine schedule;
- 128 records per update;
- CODI's answer-position distillation, alongside DSA's `<SA>` alignment.

The default arm makes **no anchor writes**, so the first question is only whether the latent loop
works: does it beat one loop and get off the 3–5% floor of the earlier runs?

**Before running:** Runtime → Change runtime type → **GPU** (A100 if offered, otherwise T4), then
Runtime → **Run all**. Everything persists under Drive `MyDrive/dynamic-state-anchoring/`.

Roughly, on a T4:
- preparing 32,768 training rows: 10–20 min, first time only;
- choosing a batch size: about 5 min;
- training: the notebook prints an upper bound before it starts;
- evaluation: 15–30 min.

None of these times have been measured on a GPU yet.

**If Colab disconnects:** reconnect to the same GPU type and Run all again. The run resumes from its
last Drive backup (about every 20 minutes) at the same code commit. A different GPU type cannot
resume a started run; set `RUN_TAG` to start a new one.

In [ ]:
# A run is identified by these settings; rerunning with the same values resumes it.
TRAIN_RECORDS = 32_768  # one pass at 128 records per update = 256 updates
WRITE_MODE = "static"   # no anchor writes; or "learned", "every_loop", "periodic"
STEPS_JOIN = "space"    # "newline": one reasoning step per line, the format the checkpoint learned
NEW_MODULE_LR = None    # None: the anchor, workspace, controller and projection also use 3e-3
RUN_TAG = ""            # e.g. "-b" for a fresh run with the same settings
REPO_REF = "main"       # code for a new run; a started run always resumes at its own commit
assert TRAIN_RECORDS % 128 == 0, "use a multiple of 128 records"

In [ ]:
from pathlib import Path
import json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
RUN_NAME = f"codi-{WRITE_MODE}-{STEPS_JOIN}-n{TRAIN_RECORDS}{RUN_TAG}"
BACKUP_DIR = DRIVE / "codi-recipe" / RUN_NAME
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PLAN = BACKUP_DIR / "plan.json"
ref = json.loads(PLAN.read_text())["commit"] if PLAN.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print("Code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = str(DRIVE / "cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
RESULTS = DRIVE / "results"
for path in Path("results").rglob("*"):  # Seed Drive results with the committed rows for comparison.
    target = RESULTS / path.relative_to("results")
    if path.is_file() and not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
WORK = Path("/content/codi-recipe")  # fast local disk; checkpoints are backed up to BACKUP_DIR
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

In [ ]:
run("scripts/setup_data.py", "--eval-only")  # evaluation sets and the pinned GSM8K-Aug arrays
SMOKE = WORK / "smoke-cuda.json"
run("scripts/smoke_pretrained_dsa.py", "--device", "cuda", "--output", SMOKE, "--max-new-tokens", 4)
# A seeded sample of the training split, prepared once on Drive and reused afterwards.
PREPARED_DRIVE = DRIVE / "prepared" / f"codi-train-{TRAIN_RECORDS}-{STEPS_JOIN}"
run("scripts/prepare_dsa_pilot.py", "--source-dir", os.environ["DSA_DATA_DIR"], "--output-dir", PREPARED_DRIVE,
    "--train-limit", TRAIN_RECORDS, "--loops", 8, "--seed", 7, "--steps-join", STEPS_JOIN)
PREPARED = WORK / PREPARED_DRIVE.name
shutil.copytree(PREPARED_DRIVE, PREPARED, dirs_exist_ok=True)

In [ ]:
# Picks a batch size that fits (first run only), trains with Drive backups, then evaluates
# 8 loops, 1 loop and the explicit route on dev64 and the 500 validation questions.
NAME = f"dsa-codi-{WRITE_MODE}-n{TRAIN_RECORDS // 1024}k" + ("-newline" if STEPS_JOIN == "newline" else "") + RUN_TAG
run("scripts/run_codi_recipe.py", "--prepared-dir", PREPARED, "--work-dir", WORK / RUN_NAME,
    "--backup-dir", BACKUP_DIR, "--results-dir", RESULTS, "--smoke-report", SMOKE, "--device", "cuda",
    "--name", NAME, "--write-mode", WRITE_MODE,
    *([] if NEW_MODULE_LR is None else ["--new-module-learning-rate", NEW_MODULE_LR]))

In [ ]:
run("scripts/evaluate.py", "--table", "--results-dir", RESULTS)

**Reading the result**
- `-k8` against `-k1` is the first check: the summary above gives the paired counts and an exact
  p-value. The earlier no-write run scored 3.1% on dev64 and 4.6% on validation, with 1 loop as good as 8.
- `-explicit` should stay near `gpt2-sft`, the checkpoint before DSA training: 42.4% on validation.
  LoRA freezes the base weights, so a large drop here would be surprising.
- For comparison, `codi` scores 41.2% on validation.

**Sharing:** the new rows are under Drive `results/runs/` and `results/predictions/`. Copy those folders
into your clone's `results/` and commit, or point Claude at them.